# 06. 클래스 / OOP

인스턴스 vs 클래스 속성, 매직 메서드, 상속, `@dataclass`까지. 자율주행/로봇 쪽에서 익숙한 '센서 메시지' 예제로 풀어봅니다.

## 클래스 기본: __init__, 인스턴스 속성 vs 클래스 속성

In [1]:
class SensorReading:
    unit = "m/s"  # 클래스 속성: 모든 인스턴스가 공유

    def __init__(self, name, value):
        self.name = name    # 인스턴스 속성: 개별 객체마다 다름
        self.value = value

r1 = SensorReading("lidar", 12.3)
r2 = SensorReading("radar", 11.8)

print(r1.name, r1.value, r1.unit)
print(r2.name, r2.value, r2.unit)

SensorReading.unit = "km/h"  # 클래스 속성을 바꾸면 모든 인스턴스에 반영
print(r1.unit, r2.unit)


lidar 12.3 m/s
radar 11.8 m/s
km/h km/h


## 매직 메서드 (dunder method)

`__repr__`, `__eq__` 등을 구현하면 내장 함수/연산자와 자연스럽게 어울리는 객체를 만들 수 있습니다.

In [2]:
class Vector2D:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):  # 디버깅용 표현 (REPL/print에서 보이는 모습)
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __add__(self, other):  # + 연산자 오버로딩
        return Vector2D(self.x + other.x, self.y + other.y)

    def __len__(self):  # len()으로 벡터 크기(정수로 근사) 흉내
        return int((self.x**2 + self.y**2) ** 0.5)

v1 = Vector2D(1, 2)
v2 = Vector2D(3, 4)
print(v1 + v2)
print(v1 == Vector2D(1, 2))
print(len(v2))


Vector2D(4, 6)
True
5


### 자주 쓰는 매직 메서드 ↔ 실제 호출 매핑

파이썬 문법(연산자, 내장 함수)은 결국 아래처럼 해당 매직 메서드 호출로 번역됩니다. 이 매핑을 알아두면 "왜 `len(obj)`가 되지?" 같은 걸 외우지 않고 유추할 수 있습니다.

| 코드 | 실제로 호출되는 것 |
|---|---|
| `for x in obj:` | `obj.__iter__()`, 그 다음 `__next__()` 반복 |
| `len(obj)` | `obj.__len__()` |
| `obj[i]` | `obj.__getitem__(i)` |
| `MyClass(...)` | `obj.__init__(...)` (객체 생성 직후 호출) |
| `print(obj)` | `obj.__str__()` |
| `obj1 + obj2` | `obj1.__add__(obj2)` |
| `obj1 == obj2` | `obj1.__eq__(obj2)` |

In [3]:
class Bag:
    def __init__(self, items):
        self._items = list(items)

    def __getitem__(self, index):   # obj[i] -> 이 메서드로 번역됨
        return self._items[index]

    def __len__(self):               # len(obj) -> 이 메서드로 번역됨
        return len(self._items)

bag = Bag(["lidar", "radar", "camera"])
print(bag[0])       # Bag.__getitem__(bag, 0)
print(len(bag))      # Bag.__len__(bag)

# __getitem__만 구현해도 for문이 동작함 (0부터 IndexError가 날 때까지 순회)
for item in bag:
    print("item:", item)


lidar
3
item: lidar
item: radar
item: camera


## @property: 메서드를 속성처럼

값 검증이나 계산된 속성을 넣고 싶을 때, 외부 인터페이스(`obj.value`)는 그대로 유지하면서 내부 구현만 바꿀 수 있습니다.

In [4]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("절대영도보다 낮을 수 없습니다")
        self._celsius = value

    @property
    def fahrenheit(self):  # 읽기 전용 계산 속성
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.fahrenheit)
t.celsius = 30
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as e:
    print("에러:", e)


77.0
86.0
에러: 절대영도보다 낮을 수 없습니다


## 상속과 super()

In [5]:
class Sensor:
    def __init__(self, name):
        self.name = name

    def read(self):
        raise NotImplementedError  # 서브클래스가 반드시 구현해야 함을 명시

class LidarSensor(Sensor):
    def __init__(self, name, range_m):
        super().__init__(name)  # 부모 __init__ 호출
        self.range_m = range_m

    def read(self):
        return f"{self.name} lidar reading within {self.range_m}m"

s = LidarSensor("front-lidar", 50)
print(s.read())
print(isinstance(s, Sensor))


front-lidar lidar reading within 50m
True


## @dataclass: 보일러플레이트 없는 데이터 클래스

`__init__`, `__repr__`, `__eq__`를 자동 생성해줍니다. '값을 담는 용도'의 클래스라면 거의 항상 이게 낫습니다.

In [6]:
from dataclasses import dataclass, field

@dataclass
class DetectedObject:
    label: str
    confidence: float
    bbox: tuple = field(default=(0, 0, 0, 0))

obj1 = DetectedObject("car", 0.92, (10, 20, 100, 80))
obj2 = DetectedObject("car", 0.92, (10, 20, 100, 80))

print(obj1)                 # __repr__ 자동 생성됨
print(obj1 == obj2)         # __eq__도 자동 생성됨 (값 비교)


DetectedObject(label='car', confidence=0.92, bbox=(10, 20, 100, 80))
True


## 헷갈렸던 점 / 더 찾아볼 것

- `@dataclass(frozen=True)`로 불변 데이터클래스도 가능 — 설정값 객체에 유용
- 매직 메서드는 이 외에도 `__getitem__`, `__iter__`(07번과 연결), `__enter__`/`__exit__`(08번과 연결)까지 다양함